# STAT 450: Case Studies in Statistics

## Mixed Effects Models

For background on the data set, see class notes and
 https://www.kaggle.com/ojwatson/mixed-models



First step:  Install:  lme4  and load libraries, read in the data file

In [ ]:
# install.packages("lme4")
library(tidyverse)
library(lme4)
bounce_data <- read.csv(file="data/bounce_speed.csv", header=TRUE)
head(bounce_data)
dim(bounce_data)


<span style="color:red; font-weight:bold;">Exercise:</span>  All on one plot, plot Bounce Time versus Age, colour coded by county.  

In [ ]:
### Remove the initial #s  and insert your code in the ...
#bounce_data  |>  ggplot(aes(x=...,y=...,colour=...)) +
#              geom_point() +
#              xlab("age (years) ") + ylab("Bounce Time (sec)")

<span style="color:red; font-weight:bold;">Exercise:</span>   Make separate plots for each county, colour-coding by location.  Add the least squares line (one line per county). 

In [ ]:
### Remove the initial #s  and insert your code in the ...
#bounce_data  |>  ggplot(aes(x=age,y=bounce_time,colour=...)) +
#            geom_point(size=.5) +
#            facet_wrap(.~...) +
#            geom_smooth(method = "...", se=FALSE, color="black", formula = y ~ x, fullrange = T,lwd=.5) +
#              xlab("age (years) ") + ylab("Bounce Time (sec)")

<span style="color:red; font-weight:bold;">Discussion:</span> What do you notice in these plots?  Do you think separate lines are needed?

Here is a very nice way to look at the intercept and slope estimates of the 8 lines.

In [ ]:
fit_lm=lmList(bounce_time~age|county,bounce_data)
plot(confint(fit_lm))

###  Fit a mixed effects model with random intercept but non-random (fixed) slope
<span style="color:red; font-weight:bold;">Discussion of input and output:</span> 
- Look at the formula in the `lmer` call.
- From the summary, identify what is random, what is fixed.
- What is the estimate of the variance of the random intercept?
- What are the estimates of the fixed effect parameters?
- What is the fitted model?  that is what is $\hat{B}_{c,i} = ??$


In [ ]:
lme_int <- lmer(bounce_time ~  (1|county) + age, data=bounce_data)  
summary(lme_int)

We can **extract the fixed effects**.

In [ ]:
fixef(lme_int)

We can **extract the random effects**.

In [ ]:
ranef(lme_int)

**Check X and Z.** For this random intercept and fixed slope model, find the fixed effects model matrix, X, and the random effects model matrix, Z.  Do they make sense?

**Check X** 

In [ ]:
Xmat <- getME(lme_int,"X")
dim(Xmat)
head(Xmat)
tail(Xmat)

Double-check column 1 (many ways to do this)

In [ ]:
table(Xmat[,1])

**Check Z.**
  Harder to look at!  
- Consider the dimension.  Do the number of columns make sense?
- Maybe we can table the values in Z.  Do the counts make sense?

In [ ]:
Zmat <- getME(lme_int,"Z")
dim(Zmat)
head(Zmat)
tail(Zmat)

In [ ]:
table(as.vector(Zmat))

###  Fit a mixed effects model with random intercept and random slope

In [ ]:
lme_slope <- lmer(bounce_time ~  age +  (age|county), data=bounce_data)  
lme_slope

**Convergence issue!** Sometimes scaling the x-value helps.

###  Second try to fit a mixed effects model with random intercept and random slope - scaled ages

In [ ]:
data_scaled <- bounce_data |>  mutate(scaled_age = (age-mean(age))/sd(age))
head(data_scaled)

In [ ]:
data_scaled  |>  ggplot(aes(x=scaled_age,y=bounce_time,colour=location)) +
            geom_point(size=.5) +
            facet_wrap(.~county) +
            geom_smooth(method = "lm", se=FALSE, color="black", formula = y ~ x, fullrange = T,lwd=.5) +
              xlab("age (years) ") + ylab("Bounce Time (sec)")

<span style="color:red; font-weight:bold;">Discussion of the following input and output:</span> 
- Look at the formula in the `lmer` call.
- From the summary, identify what is random, what is fixed.
- What is the estimate of the variance of the random intercept?  other variance estimates?
- What are the estimates of the fixed effect parameters?
- What is the fitted model?  that is what is $\hat{B}_{c,i} = ??$

In [ ]:
lme_slope <- lmer(bounce_time ~  scaled_age +  (scaled_age|county), data=data_scaled)  
lme_slope

<span style="color:red; font-weight:bold;">Exercise:</span>    Extract the fixed effects and the random effects to make sure you understand the model.

In [ ]:
### enter your  code here


<span style="color:red; font-weight:bold;">Exercise:</span>  In Cheshire county, what is the estimated regression line?

$\hat{B} = ~?? ~  + ~?? ~\times$ scaled_age?

**Are you wondering...** Why did I write `bounce_time ~  scaled_age +  (scaled_age|county)` instead of 
`bounce_time ~  scaled_age|county`?

Let's see: try just  `(scaled_age|county)`: 
- What is the model $\hat{B}_{c,i} = ??$

In [ ]:
lme_slope2 <- lmer(bounce_time ~  (scaled_age|county), data=data_scaled)  
lme_slope2

In [ ]:
fixef(lme_slope2)
ranef(lme_slope2)

### Does bounce time depend on age (in England)?  


Recall the full model for `lme_slope`: `bounce_time ~ scaled_age|county + (scaled_age|county)`

$ B_{c,i} =   a  ~+  ~ \alpha^*_c ~ + ~ (b + \beta^*_c )~     x_{c,i} ~+ ~\epsilon_{c,i},
$
 
 Test:  $H_o: b = 0$.
 
 We have fit via `lme_slope2`:  `bounce_time ~  (scaled_age|county)`

$
 B_{c,i} =  a  ~+  ~ \alpha^*_c ~ + ~  \beta^*_c ~     x_{c,i} ~+ ~\epsilon_{c,i}
$

Use `anova` to compare. 
- `lme_slope2`:  `bounce_time ~  (scaled_age|county)`
-  `lme_slope`: `bounce_time ~ scaled_age|county + (scaled_age|county)`

In [ ]:
anova(lme_slope2,lme_slope)

**Conclusion:  is bounce time related to age in England?**

Why not the following model as the null model?  In some ways ....

In [ ]:
 lme_no_slope <-  lmer(bounce_time ~ 1 + (1|county),data=data_scaled)
summary(lme_no_slope)

### NESTING in a Mixed Effect Model  (if there is time - else, read on your own)

With our data set, locations a, b and c are randomly chosen with no meaning, e.g. we don't have 
a=urban, b=rural, c=suburban.

So a, b and c are nested within county.

**IF** a, b and c had had meaning (e.g. a=urban, b=rural, c=suburban), then this would be a crossed design, not nested.

Full nested model -  for bounce time of person $i$ in county $c$, location $l$:

$ B_{c,l,i} =   a  ~+  ~ \alpha^*_c ~+~ \alpha^{**}_l  + ~ (b + \beta^*_c ~ +~  \beta^{**}_l )~  x_{c,i} ~+ ~\epsilon_{c,l,i}
$ 

Now try adding location as nested within county.  Try for random intercept, non-random slope.

In [ ]:
lme_int_nested <- lmer(bounce_time ~  (1|county/location) + scaled_age,data=data_scaled )
lme_int_nested

In [ ]:
fixef(lme_int_nested)
ranef(lme_int_nested)

Is it important to include random intercepts for locations within counties? 

In [ ]:
lme_int_county <- lmer(bounce_time ~  (1|county) + scaled_age,data=data_scaled )
lme_int_county

In [ ]:
anova(lme_int_county,lme_int_nested)

**Try to nest location within county for both slope and intercept.** **Convergence trouble!** 
Too complicated a model for the data?  Note Corr = -1.00 in summary.  Hmmm....  This signals the dataset is causing us a problem.  Complicated "fixes"!

    

In [ ]:
lme_all_nested <- lmer(bounce_time ~  scaled_age + (scaled_age|county/location),data=data_scaled )
lme_all_nested


In [ ]:
fixef(lme_all_nested)

Not liking what I see with the random effects.

In [ ]:
ranef(lme_all_nested)
plot( (ranef(lme_all_nested))$`location:county`,xlab="Intercept",ylab="Slope")
title("Random Effects for location:county")